# Swiss German LoRA fine-tuning on Colab (Q3)
Runtime → Change runtime type → **T4 GPU**, then run the cells in order

STT4SG-350 is licensed and not downloaded here, so the training data comes from your own Google Drive:

- `colab_package.zip`: the repository's `src/` and `eval_sets/` (upload in step 2)
- `stt4sg_finetune.zip` in `MyDrive/`: the manifest and the 16 kHz train and dev subsets, made locally with
  `python src/prepare_stt4sg.py --stt4sg-dir path/to/STT4SG-350 --test-size 5000 --train-size 5000 --dev-size 300`
  and zipped with the command in the README
- `gsw_transcribe.zip` in `MyDrive/`: the 3,515 test clips (see `transcribe_colab_gsw.ipynb`), used in step 6

Results go to Google Drive: the model to `MyDrive/small-lora-gsw/`, the transcripts to `MyDrive/gsw_results_finetuned/`
Copy the transcripts to `results/finetuned/` and continue with step 5b of the README

In [ ]:
!nvidia-smi | head -15

## 1. Environment (same versions as the laptop)

In [ ]:
# Colab ships torchao 0.10, which this peft version rejects; we do not use it
!pip -q uninstall -y torchao
!pip -q install transformers==5.22.1 peft==0.21.2 ctranslate2==4.8.2 faster-whisper whisper-normalizer jiwer soundfile huggingface_hub tabulate pandas
# ctranslate2 needs the CUDA 12 cuBLAS and cuDNN libraries, which the current Colab image no longer ships
!pip -q install nvidia-cublas-cu12 "nvidia-cudnn-cu12==9.*"
import os, nvidia.cublas.lib, nvidia.cudnn.lib
libs = [list(nvidia.cublas.lib.__path__)[0], list(nvidia.cudnn.lib.__path__)[0]]
os.environ["LD_LIBRARY_PATH"] = ":".join(libs + [os.environ.get("LD_LIBRARY_PATH", "")])

## 2. Upload `colab_package.zip` and unpack it

In [ ]:
from google.colab import files
uploaded = files.upload()   # pick colab_package.zip
!mkdir -p /content/project && cd /content/project && unzip -oq /content/colab_package.zip

## 3. Train and dev subsets from Google Drive
Copied to the Colab disk first because reading thousands of small files from Drive is slow

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!cp /content/drive/MyDrive/stt4sg_finetune.zip /content/
%cd /content/project
!unzip -oq /content/stt4sg_finetune.zip && rm /content/stt4sg_finetune.zip

In [ ]:
# Zips made in Finder put everything inside an extra folder: move its contents up
import os, glob
for d in glob.glob('/content/project/*'):
    inner = os.path.isdir(os.path.join(d, 'src')) or os.path.isdir(os.path.join(d, 'data'))
    if os.path.isdir(d) and inner and os.path.basename(d) not in ('src', 'data', 'eval_sets', 'ratings', 'results'):
        !cp -r "{d}"/. /content/project/ && rm -rf "{d}"
!rm -rf /content/project/__MACOSX; ls /content/project

In [ ]:
# Check: train and dev rows with audio and all files present (expected: train 4998, dev 300, 0 missing)
import pandas as pd
from pathlib import Path
m = pd.read_csv('data/manifests/stt4sg_gsw.csv', keep_default_na=False)
a = m[(m.audio_path != '') & (m.split != 'test')]
print(a.groupby('split').agg(recordings=('id', 'size'), hours=('duration', lambda s: round(s.sum() / 3600, 2))))
print('missing train/dev files:', sum(not Path(p).exists() for p in a.audio_path))

## 4. Smoke test (about 2 minutes)
Whisper has no language code for Swiss German, so training and decoding use `de` as in the zero-shot baseline

In [ ]:
!python src/finetune_lora.py --manifest data/manifests/stt4sg_gsw.csv --whisper-language de --name smoke --max-train 64 --epochs 1 --dev-wer-items 0 2>&1 | grep -v -i warn | tail -8

## 5. The real run (3 epochs, about 35 minutes on a T4)
4,972 training recordings (6.9 h) after removing 26 whose sentence is held out, 300 dev recordings
Progress is printed every 10 steps, `models/small-lora-gsw/training_log.csv` gets a row per epoch

In [ ]:
!rm -rf models/smoke
!python -u src/finetune_lora.py --manifest data/manifests/stt4sg_gsw.csv --whisper-language de --name small-lora-gsw 2>&1 | grep -v -i warn

In [ ]:
# Save the model to Google Drive right away: the Colab disk is lost when the session ends
!mkdir -p /content/drive/MyDrive/small-lora-gsw
!cp -r models/small-lora-gsw/ct2 models/small-lora-gsw/adapter models/small-lora-gsw/training_log.csv models/small-lora-gsw/training_config.json /content/drive/MyDrive/small-lora-gsw/
!cat models/small-lora-gsw/training_log.csv

## 6. Transcribe the 3,515 test clips with the fine-tuned model
Same decoder and float16 GPU precision as the zero-shot Swiss German baseline, so the two are directly comparable

In [ ]:
# Test clips from gsw_transcribe.zip (works with or without the extra Finder folder)
!rm -rf /content/tx && mkdir -p /content/tx && cd /content/tx && unzip -oq /content/drive/MyDrive/gsw_transcribe.zip
import glob
test_dir = [d for d in glob.glob('/content/tx/**/data/raw/stt4sg/test', recursive=True) if '__MACOSX' not in d][0]
!mkdir -p data/raw/stt4sg/test && cp --update=none "{test_dir}"/*.wav data/raw/stt4sg/test/
!ls data/raw/stt4sg/test | wc -l

In [ ]:
!python src/transcribe.py --manifest data/manifests/stt4sg_gsw.csv --whisper-language de --compute-type float16 \
    --models small-lora-gsw=models/small-lora-gsw/ct2 --out-dir /content/drive/MyDrive/gsw_results_finetuned

In [ ]:
r = pd.read_csv('/content/drive/MyDrive/gsw_results_finetuned/small-lora-gsw_gsw.csv', keep_default_na=False)
print(len(r), 'rows,', (r.hypothesis.str.strip() == '').sum(), 'empty, model', r.model.iloc[0], ', compute_type', r.compute_type.iloc[0])

When done: Runtime → Disconnect and delete runtime. Everything needed is on Google Drive